# 05a - Chatbot (Module 3) benchmark: build the prompts

43 conversations (53 turns) in six groups - follow-up questions that need memory, Kannada and Kanglish, leaf photos and
past scans, weather with/without a location, greetings, prompt injection. Speech input is **not** evaluated (it needs
real recordings).

Systems, all answered by the same LLM so only the chatbot pipeline differs:
* **PQ** - PlantIQ chatbot (`plantiq_chat_snapshot`, checked identical to backend `chat.process_message` by
  `verify_snapshot.py`), real hybrid retriever, real ResNet-50 on the photos, recorded Open-Meteo weather.
* **SK** - Selva Kumar et al. as a LangChain ConversationalRetrievalChain (condense follow-up -> dense top-4 -> default
  QA prompt), YOLOv8s-cls baseline (fold 0) for photos.
* **PQoff** - PlantIQ with no LLM reachable (heuristic router + built-in offline reply).

LLM steps (router, LangChain condense step, answers) are read from `05_chat/answers/` (README). A turn needs the
answers of the earlier turns as history, so this notebook is run again after each turn depth is answered.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "05_chat", ROOT / "03_rag"):
    sys.path.insert(0, str(p))

import json
import subprocess

from common import config, env

env.require_cuda()
for script in ("make_cases.py", "make_router_outputs.py"):
    print(subprocess.run([sys.executable, str(ROOT / "05_chat" / script)], capture_output=True, text=True,
                         encoding="utf-8").stdout.strip())

In [ ]:
import build_prompts

models = build_prompts.main()

In [ ]:
import pandas as pd

rows = pd.read_json(ROOT / "05_chat" / "answers" / "chat_inputs.jsonl", lines=True)
pq = rows[rows.system == "PQ"]
print("CNN on the photos:", pq.dropna(subset=["diagnosis"]).apply(lambda r: (r.case, r.diagnosis["class"], r.diagnosis["confidence"]), axis=1).tolist())
print("retrieval skipped on:", pq[pq.contexts.str.len() == 0].case.tolist())
print("weather detected on:", pq[pq.weather_query.astype(bool)].case.tolist())
rows.groupby("system").contexts.apply(lambda c: c.str.len().mean()).rename("passages per turn")